In [ ]:
using Gridap.ODEs
using Gridap        
using GridapGmsh
using Gridap.Geometry
using Gridap.TensorValues
using Plots
using LinearAlgebra
using  Gridap.Fields
using  Gridap.CellData
using  Gridap.ReferenceFEs  
using  Gridap.Fields
using Random
using LinearAlgebra
using Gridap.Arrays
using Base.Iterators
using LineSearches: BackTracking

In [ ]:
const E = 190e9
const ν = 0.3
const G = E/(2*(1+ν))
const λ_ps = (E*ν)/((1+ν)*(1-2*ν)) 
const μ = G
const λ = λ_ps*(2*μ/(λ_ps+2*μ))
ρ = 8000.0
 G0 = 5e6
 fₜ = (2812.25e6)
r = 3/1000
const E_0_dot1 = 1.0e-6
const c1 = 0.055

In [ ]:
model = GmshDiscreteModel("KW_sym_ver.msh")
writevtk(model,"KW_sym_ver")

In [ ]:
order = 1
reffeG = ReferenceFE(lagrangian,VectorValue{1,Float64},order)
VG = TestFESpace(model,reffeG;
          conformity=:H1)
order = 1
reffephi  = ReferenceFE(lagrangian,Float64,order)
Vphi  = TestFESpace(model,reffephi;
          conformity=:H1)
Uphi = TrialFESpace(Vphi)
f_new = 1.0

In [ ]:
order = 1
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
V = TestFESpace(model,reffe;
          conformity=:H1,  dirichlet_tags=["BottomEdge","LoadEdge","RightEdge"],
          dirichlet_masks=[(false,true),(true,false),(true,false)]) 

In [ ]:
Ω = Triangulation(model)
degree = 2*order
dΩ = Measure(Ω,degree)

In [ ]:
labels = get_face_labeling(model)
LoadTagId = get_tag_from_name(labels,"LoadEdge")
Γ_Load = BoundaryTriangulation(model,tags = LoadTagId)
dΓ_Load = Measure(Γ_Load,degree)
n_Γ_Load = get_normal_vector(Γ_Load)

In [ ]:
Gr = get_grid(model)
nodes = get_node_coordinates(Gr)
Nₑ, Nₙ = num_cells(model), num_nodes(model)
nodeCoordX, nodeCoordY = [nodes[i][1] for i in 1:Nₙ], [nodes[i][2] for i in 1:Nₙ]
elem = get_cell_node_ids(Gr)

In [ ]:
dΩ_ro = Measure(Ω,1)
x_S = get_cell_points(dΩ_ro)

In [ ]:
using NearestNeighbors
data = zeros(2,Nₙ)
data[1,:] =nodeCoordX
data[2,:] =nodeCoordY
points = data
balltree = BallTree(data)
idxs = inrange(balltree, points, r, true)

In [ ]:
function G_kill(σ_eq,dot_σ_eq,fₜ)
 G_kill = 0.25*G0.*(( (σ_eq./fₜ -1)+ abs∘(σ_eq./fₜ - 1))).*((dot_σ_eq)+ abs∘(dot_σ_eq))
    return G_kill
end

In [ ]:
function new_EnergyState(ψPlusPrev_in,t_s_in,ψhPos_in)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
         tPlus_out = t_s_in
        damaged = false
    else
        ψPlus_out = ψPlus_in
        tPlus_out = T
        damaged = true
    end
    damaged, ψPlus_out, tPlus_out   
end


function new_EnergyStateAvg(ψPlusPrev_in,ψhPos_in,ψhPos_prev)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
        damaged = false
    else
        ψPlus_out = 0.5*(ψPlus_in + ψhPos_prev)
        damaged = true
    end
    damaged, ψPlus_out   
end

In [ ]:
σ(ε_nl) =  λ*tr(ε_nl)*one(ε_nl) + 2*μ*ε_nl

In [ ]:
function σ_eq(ε, ε_nl)
    εArray, εArray_nl = get_array.((ε, ε_nl))
    Λ, P = eigen(εArray)
    Λ_nl, P_nl = eigen(εArray_nl)
    Λpos = diagm(0 => max.(0, Λ))
    Λpos_nl = diagm(0 => max.(0, Λ_nl))
    εPos = TensorValue(P * Λpos * P')
    εPos_nl = TensorValue(P_nl * Λpos_nl * P_nl')
    ψPos = 0.5 * ((tr(ε_nl) >= 0) * (λ * tr(ε_nl) * tr(ε_nl)) + 2μ * (εPos_nl ⊙ εPos_nl))
    return √(2 * ψPos * E)
end

In [ ]:
function nonLocalGk(G_k_prev,t_k_prev)
    GkVec = evaluate(G_k_prev,x_S)
    TkVec = evaluate(t_k_prev,x_S)
    caches = [array_cache(GkVec) for k in 1:Threads.nthreads()]
    caches_T = [array_cache(TkVec) for k in 1:Threads.nthreads()]
    Gk_nds = zeros(Nₙ)
    Tk_nds = zeros(Nₙ)
    Gk_sum = zeros(Nₙ)
   Tk_sum = zeros(Nₙ)
    Gk_count = zeros(Int, Nₙ)
    Threads.@threads for iel in 1:Nₑ
        cache = caches[Threads.threadid()]
        cache_T = caches_T[Threads.threadid()]
        ElNdInd = elem[iel]
    val_G = getindex!(cache, GkVec, iel)
    val_T = getindex!(cache_T, TkVec, iel)
for node in ElNdInd
        Gk_sum[node] += val_G[1]
        Tk_sum[node] += val_T[1]
        Gk_count[node] += 1
        end
    end
Gk_nds = Gk_sum ./ Gk_count
Tk_nds = Tk_sum ./ Gk_count
  Gk_nds_NL = zeros(Nₙ)
    Threads.@threads for nd_id in 1:Nₙ
        NeighHood = idxs[nd_id]
        @inbounds Gk_nds_NL[nd_id] = (sum(Gk_nds[NeighHood]) ) / (length(NeighHood))
    end
    return Gk_nds_NL,Tk_nds
end

In [ ]:
function nonlocalfield(Gk_nds, t)
    ϕVec_st = similar(Gk_nds)  
    @. ϕVec_st = exp(-(Gk_nds * t)) 
    return FEFunction(Uphi, ϕVec_st)
end

In [ ]:
function project(q,model,dΩ,order)
  reffe = ReferenceFE(lagrangian,Float64,order)
  V = FESpace(model,reffe,conformity=:L2)
  a(u,v) = ∫( u*v )*dΩ
  l(v) = ∫( v*q )*dΩ
  op = AffineFEOperator(a,l,V,V)
  qh = solve(op)
  qh
end

function project_vector(q,model,dΩ,order)
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
V = TestFESpace(model,reffe;
          conformity=:H1)
  a(u,v) = ∫( u⋅v )*dΩ
  l(v) = ∫( v⋅q )*dΩ
  op = AffineFEOperator(a,l,V,V)
  qh = solve(op)
  qh
end

In [ ]:
function E_dot(ε_new,ε_old)
    εArray, εArray_nl = get_array.((ε_new,ε_old))
    Λ, P = eigen(εArray)
    Λ_nl, P_nl = eigen(εArray_nl)
    e1 = Λ[1]
    e2 = Λ[2]
    e_nl1 = Λ_nl[1]
    e_nl2 = Λ_nl[2]
     Λ_new = max(e1,e2)
     Λ_old = max(e_nl1,e_nl2)
    E_dot = (1/T)*(Λ_new-Λ_old)
end

In [ ]:
function RateDepGc(_E_dot)
    if _E_dot > E_0_dot1
     ft2 = fₜ*(1+((_E_dot/E_0_dot1)^c1 - 1 ))
    else 
         ft2 = fₜ
    end
    return  ft2
end

## constant average acceleration method

In [ ]:
function step_disp(uApp,dot_σ_eq,uh,vh,ah,G_k_cell,ϕ,dt,t_cell,dot_E,uh_upd,G_k_prev,trac_top,trac)
           
solver = LUSolver()  
u1(x) = VectorValue(0.0,0.0)
u2(x) = VectorValue(uApp,0.0)
u3(x) = VectorValue(0.0,0.0)
U = TrialFESpace(V,[u1,u2,u3])
σ_eq_s = σ_eq∘((ε(uh_upd))*ϕ,(ε(uh_upd))*ϕ)
ft_new= fₜ 
G_k_in = G_kill(σ_eq_s,dot_σ_eq,ft_new)
update_state!(new_EnergyState,G_k_cell,t_cell,G_k_in)
G_k_nd_nl,t_nds = nonLocalGk(G_k_cell,t_cell)
ϕ = (nonlocalfield(G_k_nd_nl,t_nds))
    m(t, u, v) = ∫(ρ*v⋅(∂t(∂t(u))))dΩ
      a(t,u,v) = ∫( (ε(v)) ⊙ ((σ∘((ε(u))*(ϕ)))*(ϕ)))*dΩ
res(t, u, v) = m(t, u, v) + a(t, u, v) 
ls = LUSolver()
nl_solver = NLSolver(ls, method=:newton, iterations=1, show_trace=true)
order = 2
op = TransientFEOperator(res,U,V;order)
t0 = T-dt
tF = T
γ = 0.5
β = 0.25   
ode_solver = Newmark(nl_solver,dt,γ,β)
sol_t = solve(ode_solver,op,t0,tF,(uh,vh,ah))
u_new = zero(V)
        for (tn,uhn) in sol_t
            u_new = uhn
        end
return u_new,ϕ,G_k_cell,t_cell, G_k_nd_nl, ft_new, t_nds, cache
end

In [ ]:
cd("KW_new_energy_19_06_25_copy1")

In [ ]:
ft_new =interpolate_everywhere(fₜ,Vphi)
G_k_cell = CellState(0.0,dΩ_ro)
T= 0.0
g= 0.0
t_cell = CellState(T,dΩ_ro)
uApp = 0.0
vApp = 16.5
delT = 2e-7
Tmax = 1e-4
Load = Float64[]
Displacement = Float64[]
push!(Load, 0.0)
push!(Displacement, 0.0)
trac_top = VectorValue(0.0,1.0e6)
trac = VectorValue(0.0,-1.0e6)
count_n = 0
count_inner = 0
uh = zero(V)
dt=delT
uh_in_FE = zero(V)
 uh_iter = zero(V)
uh_iter_p = zero(V)
vh_iter_cell = CellState(VectorValue(0.0,0.0),dΩ)
ah_iter_cell = CellState(VectorValue(0.0,0.0),dΩ)
v1(x) = VectorValue(0.0,0.0)
v2(x) = VectorValue(vApp,0.0) 
v3(x) = VectorValue(0.0,0.0)
a2(x) =VectorValue(0.0,-g) 
Uv = TrialFESpace(V,[v1,v2,v3])
Ua = TrialFESpace(V,[v1,a2,v3])
vh_vec = get_free_dof_values(uh)
vh_iter = FEFunction(Uv,vh_vec)
vh_iter_p = vh_iter
ah_vec = get_free_dof_values(uh)
ah_iter = FEFunction(Ua,ah_vec)
ah_iter_p = ah_iter
dot_σ_eq = (σ_eq∘(ε(uh),(ε(uh))) - σ_eq∘(ε(uh_in_FE),(ε(uh_in_FE))))./delT
G_k_cell = CellState(0.0,dΩ_ro)
G_k_prev = zero(Vphi)
innerMax = 20
dot_E = E_dot∘(ε(uh),ε(uh_in_FE))
G_k_nd_nl,t_nds = nonLocalGk(G_k_cell,t_cell)
ϕ = interpolate_everywhere(f_new,Uphi)
σ_eq_s = σ_eq∘(ε(uh),(ε(uh)))
cache = nothing
while T < Tmax 
        T = T + delT
    count_n = count_n +1
    uApp  = T*vApp
    uh_iter = uh
ah_vec = get_free_dof_values(ah_iter)
ah_vec = (4/(delT*delT))*(get_free_dof_values(uh_iter)-get_free_dof_values(uh_iter_p)-delT*get_free_dof_values(vh_iter))-ah_vec 
ah_iter = FEFunction(Ua,ah_vec)
vh_vec = get_free_dof_values(vh_iter)
vh_vec = vh_vec + 0.5*delT*(get_free_dof_values(ah_iter_p)+get_free_dof_values(ah_iter))  
vh_iter = FEFunction(Uv,vh_vec)
        dot_E = E_dot∘(ε(uh_iter),ε(uh_iter_p))
    uh_iter_p = uh_iter
    vh_iter_p = vh_iter
    ah_iter_p = ah_iter
    print("\n Entering displacemtent step :", float(uApp)) 
    for inner = 1:innerMax
    uh,ϕ,G_k_cell,t_cell,G_k_nd_nl,ft_new,t_nds,cache =  step_disp(uApp,dot_σ_eq,uh_iter,vh_iter,ah_iter,G_k_cell,ϕ,delT,t_cell,dot_E,uh,G_k_prev,trac_top,trac)     
    dot_σ_eq = (σ_eq∘(ε(uh)*ϕ,(ε(uh))*(ϕ+1e-6)))./T
    e = uh-uh_in_FE      
    err = sqrt(sum( ∫( e⊙e )*dΩ ))
    print("\n Error :", float(err))  
    uh_in_FE = uh
    if err <=1e-10
        break
    end
    end
    Node_Force = sum(∫( n_Γ_Load ⋅ ((σ∘( (ε(uh))*ϕ))*ϕ )) *dΓ_Load)
    push!(Load, abs(Node_Force[1]))
    push!(Displacement, T)
    if mod(count_n,20) == 0
   writevtk(Ω,"KW_Final_$count_n",cellfields=["disp"=>uh,"phi"=>ϕ,"sig"=>σ_eq∘((ε(uh))*ϕ,(ε(uh))*ϕ)])  
    end
end